In [ ]:
import numpy as np

In [ ]:
import math

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as lines

In [ ]:
dt = np.dtype([('frame', np.int32), 
               ('Rd','<i8'),('Rx',np.int16),('Ry',np.int16),
               ('Gd','<i8'),('Gx',np.int16),('Gy',np.int16),
               ('Bd','<i8'),('Bx',np.int16),('By',np.int16),
               ('rd','<i8'),('rx',np.int16),('ry',np.int16),
               ('gd','<i8'),('gx',np.int16),('gy',np.int16),
               ('bd','<i8'),('bx',np.int16),('by',np.int16),
               ('Rn',np.int32),('Gn',np.int32),('Bn',np.int32),
               ('rn',np.int32),('gn',np.int32),('bn',np.int32),
               ('stn',np.int32)]
             )

In [ ]:
intdatahalf = np.loadtxt("DroneShort1HalfDecimated.int.1",converters=float,dtype=dt)
intdatafull = np.loadtxt("DroneShort1FullDecimated.int.1",converters=float,dtype=dt)

In [ ]:
"""
For Phase1a .int file: from intdatahalf.dtype

dtype([('frame', '<i4'), 
('Rd', '<i8'), ('Rx', '<i2'), ('Ry', '<i2'), 
('Gd', '<i8'), ('Gx', '<i2'), ('Gy', '<i2'), 
('Bd', '<i8'), ('Bx', '<i2'), ('By', '<i2'), 

('rd', '<i8'), ('rx', '<i2'), ('ry', '<i2'), 
('gd', '<i8'), ('gx', '<i2'), ('gy', '<i2'), 
('bd', '<i8'), ('bx', '<i2'), ('by', '<i2'), 

('Rn', '<i4'), ('Gn', '<i4'), ('Bn', '<i4'), 
('rn', '<i4'), ('gn', '<i4'), ('bn', '<i4'), 

('stn', '<i4')])
"""
def makeDiffMats(v):
    PixIntDiffExtremal=np.vstack([v['Rd'],v['Gd'],v['Bd'],-v['rd'],-v['gd'],-v['bd']])
    NsPixInThr=np.vstack([v['Rn'],v['Gn'],v['Bn'],v['rn'],v['gn'],v['bn']])
    NPixInSThr=v['stn']
    return { 'PixIntDiffExtremal':PixIntDiffExtremal, 'NsPixInThr':NsPixInThr, 'NPixInSThr':NPixInSThr}

Ms=makeDiffMats(intdatahalf)
PIDE=Ms['PixIntDiffExtremal']
print(PIDE.shape)
print(Ms['NsPixInThr'].shape)
print(Ms['NPixInSThr'].shape)

In [ ]:
PIDE[:,0]                                                                                                                                                                                                                        

In [ ]:
PIDE[:,0].mean()

In [ ]:
PIDE[:,0]-PIDE[:,0].mean()

In [ ]:
vars=(PIDE[:,0]-PIDE[:,0].mean())*(PIDE[:,0]-PIDE[:,0].mean())
vars

In [ ]:
math.sqrt(sum(vars)/5)

In [ ]:
#np.std?
PIDE[:,0].std(ddof=1)

In [ ]:
def PIDEstd(PIDE,n):
    return PIDE[:,n].std(ddof=1)

In [ ]:
def f1(n):
    return (PIDEstd(PIDE,n))

In [ ]:
vf1=np.vectorize(f1)

In [ ]:
SkewGaussAmpl= 0.633
SkewGaussXi= 1.97
SkewGaussOmega= 1.89
SkewGaussAlpha= 2.5

SkewGauss=np.array([SkewGaussAmpl, SkewGaussXi, SkewGaussOmega, SkewGaussAlpha])
def SG(AbsStdDev):
    ans=1. - SkewGauss[0]*math.exp( -0.5*(AbsStdDev-SkewGauss[1])*(AbsStdDev-SkewGauss[1])
	    /(SkewGauss[2]*SkewGauss[2]))*( 1. + math.erf(SkewGauss[3]*(AbsStdDev-SkewGauss[1])/(SkewGauss[2]*math.sqrt(2.))) )
    return (ans)
vSG=np.vectorize(SG)

In [ ]:
def plotprob(start,finish):
    fig, ax = plt.subplots()
    xs=np.linspace(start,finish)
    ys=vSG(xs)
    #print(xs)
    #print(ys)
    ax.plot(xs,ys)
    #plt.show()
    plt.savefig("ProbFunction.jpg")


In [ ]:
plotprob(0,10)

In [ ]:
PIDNames=['Rd','Gd','Bd','rd','gd','bd']
PIDColor={ 'Rd' : (1,0,0), 'Gd' : (0,1,0), 'Bd' : (0,0,1),
    'rd' : (0,1,1), 'gd' : (1,0,1), 'bd' : (1,1,0) }

def PIDEstd(PIDE,n):
    return PIDE[:,n].std(ddof=1)

def PIDEmean(PIDE,n):
    return PIDE[:,n].mean()

def plotDiffs(intdata, fns, w,movn):
    fig, ax = plt.subplots(figsize=(14,11))
    ax.set_title(movn)
    ax.set_facecolor('black')

    for i in range(fns-1,fns+w):
        line=lines.Line2D([i+0.5,i+0.5],   [0.0,255], color='white',linewidth=0.25)
        ax.add_line(line)
        
    for name in PIDNames[0:3]:  
        ax.scatter(intdata['frame'][fns-1:fns-1+w],intdata[name][fns-1:fns-1+w],color=PIDColor[name],alpha=0.6,s=50)
    for name in PIDNames[3:6]:  
        ax.scatter(intdata['frame'][fns-1:fns-1+w],-intdata[name][fns-1:fns-1+w],color=PIDColor[name],alpha=0.8,s=15)

    Ms=makeDiffMats(intdata)
    PIDE=Ms['PixIntDiffExtremal']
    def funstd(n):
        return PIDEstd(PIDE,n-1)
    def funmean(n):
        return PIDEmean(PIDE,n-1)
    vfunstd=np.vectorize(funstd)
    vfunmean=np.vectorize(funmean) 
    xes=np.arange(fns,fns+w)
    yes=vfunstd(xes)*2
    ax.plot(xes,yes,label='std*2')
    yes=vfunmean(xes)
    ax.plot(xes,yes,label='mean')
    yes=vSG(vfunstd(xes))*100
    ax.plot(xes,yes,label='prob*100')
    ax.legend()
        
    plt.savefig(movn+str(fns)+"."+str(fns+w)+".tiff")

In [ ]:
plotDiffs(intdatahalf,110,80,"DroneShort1HalfDecimated")
plotDiffs(intdatafull,110,80,"DroneShort1FullDecimated")

In [ ]:
plotDiffs(intdatahalf,300,80,"DroneShort1HalfDecimated")
plotDiffs(intdatafull,300,80,"DroneShort1FullDecimated")

In [ ]:
plotDiffs(intdatahalf,30,410,"DroneShort1HalfDecimated")
plotDiffs(intdatafull,30,410,"DroneShort1FullDecimated")

In [ ]:
difmat[3].mean()

In [ ]:
difmat[:][0].mean()

In [ ]:
for i in range(6):
    print(i,(difmat[:][i]).mean(),(difmat[:][i].std()))

In [ ]:
difmat[0]

In [ ]:
difmat.shape

In [ ]:
difmat[:,0].std()

In [ ]:
difmat[:,1].std()

In [ ]:
difmat[:,150].std()

In [ ]:
intdatahalf[['Rd','Gd','Bd','rd','gd','bd']][0]

In [ ]:
def to6( n ):
    return "{!s:>06}".format(n)
def tothumb(n):
    return IPath+"/thumb"+to6(n)+".bmp"
tothumb(1234)

In [ ]:
width=0
height=0
def setwh(n):
    img=Image(filename=tothumb(1))
    global width
    width=img.width
    global height
    height=img.height
setwh(1)
widthTo1920=math.ceil(float(width)/1920.0)

Setup for drawing the data line.

In [ ]:
llcapx=int(width/20)
llcapy=int(height-width/20)

In [ ]:
drdbl=Drawing()
drgbl=drdbl.clone()
drdbl.font_size=30*widthTo1920
drdbl.fill_color="WHITE"
drdbl.stroke_color="WHITE"

Setup for drawing markers on Phase1a selected pixels.

In [ ]:
def M(TH) :
    return( np.array( [ [math.cos(math.pi*TH/180.), math.sin(math.pi*TH/180.)], [-math.sin(math.pi*TH/180.), math.cos(math.pi*TH/180.)] ] ) )

#Geometry of normalized unit arrows to show RGB changes
TH=30.0                  #angle of arrows away from vertical, and hands away from body
lcircr=0.1               #little circle radius
hslen=0.1                #length of each hand of an
# Unit Vectors
g1=np.array([0.0,1.0])   #unit lower case, down, green

#tiny vectors
tc=lcircr*np.array([0.0,1.0]) #radius (down, y dir of tiny circle, and foot of down unit arrow
def T(s) :
    return (tc + g1*s) #tail on tiny circle, head down by unit * s (scale, 0<=s<=1)

#Arrow body is (tc->T(s)*M..
def arrB(s) :
    return np.array([tc, T(s)])

TL=hslen*g1@M(180.0+TH) #coord of left hand rel to head
TR=hslen*g1@M(180.0-TH) #coord of right hand rel to head

#down dir Left arm LA(s) is (T(s)->TL(s))
def arrL(s) :
    return np.array([T(s), T(s)+TL])


#down dir Right arm LA(s) is (T(s)->TL(s))
def arrR(s) :
    return np.array([T(s), T(s)+TR])

def unit_up_arrow(s) :
    return -np.concat([arrB(s),arrL(s),arrR(s)])


In [ ]:
def dispdata(row):
    #print(fn, row['frame'])
    return ( [ row['Rd'],   -30.0, [row['Rx'],row['Ry']], row['Rn' ], "red" ],
             [ row['Gd'],     0.0, [row['Gx'],row['Gy']], row['Gn' ], "green" ],
             [ row['Bd'],    30.0, [row['Bx'],row['By']], row['Bn' ], "blue" ],
             [ -row['rd'], -150.0, [row['rx'],row['ry']], row['rn' ], "red" ],
             [ -row['gd'],  180.0, [row['gx'],row['gy']], row['gn' ], "green" ],
             [ -row['bd'],  150.0, [row['rx'],row['by']], row['bn' ], "blue" ] )             

In [ ]:
colvaldiv = float(128)
arrlen = 100*widthTo1920

def drdata(dwg, row):
    data = dispdata(row)
    for r in data:
        #print(r)
        #print((r[0]/colvaldiv))
        #print( "arrow", arrlen*unit_up_arrow( (r[0]/colvaldiv)) )
        line = np.round( (arrlen*unit_up_arrow(r[0]/colvaldiv))@M(r[1]) + r[2]).astype(int)
        dwg.stroke_width = 2*widthTo1920
        dwg.stroke_color = wand.color.Color( r[4] )
        for i in range(0,3):
            dwg.line(line[2*i],line[2*i+1])

In [ ]:
img=0
def vis(fn):
    global img
    img=Image(filename=tothumb(fn))
    row=np.array(intdata[fn-1])
    t=intdata[fn-1].item(0)
    intdatarowstr = str(t[0])
    for z in range(1,24,3):
        intdatarowstr+=" "+str(t[z:z+3])
    intdatarowstr+="  "+str(t[24])
    global drdbl
    draw=drdbl.clone()
    draw.text(llcapx,llcapy,intdatarowstr)
    drdata(draw, row )
    draw(img)
    #draw
    #draw(img)
    return img #so jupyter tries to print the result which makes the picture appear!      

In [ ]:
vis(79)

In [ ]:
#comment out so the .ipynb file is small enough to archive on git
#vis(319)

In [ ]:
def tophonea(n):
    return IPath+"/phonea"+to6(n)+".jpg"
def doAll():
    for i in range(nframes):
        fn=i+1
        img=vis(fn)
        print("visualized frame", fn, end="")
        img.format = 'jpeg'
        img.save(filename=tophonea(fn))
        print(" saved", fn, end="\r")

In [ ]:
doAll()